# 02 — Beijing Preprocessing

Tạo working hourly dataset. Raw giữ nguyên; working output nằm trong son_nb/output.

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import shutil

# ------------------------------------------------------------
# Robust project root finder
# Notebook lives under: Project/notebooks/son_nb/
# ------------------------------------------------------------
def find_project_root(start=None):
    start = Path(start or Path.cwd()).resolve()
    candidates = [start, *start.parents]
    for p in candidates:
        if (p / "data" / "raw").exists() and (p / "notebooks").exists():
            return p
    raise FileNotFoundError(
        "Không tìm thấy Project root. Hãy chạy notebook bên trong project."
    )

PROJECT_ROOT = find_project_root()
RAW_DIR = PROJECT_ROOT / "data" / "raw"
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
MAIN_OUTPUT = PROJECT_ROOT / "outputs"
SON_DIR = PROJECT_ROOT / "notebooks" / "son_nb"
SON_OUTPUT = SON_DIR / "output"

WORK_DATA = SON_OUTPUT / "data"
WORK_FIG = SON_OUTPUT / "figures"
WORK_TABLE = SON_OUTPUT / "tables"
WORK_PRED = SON_OUTPUT / "predictions"
WORK_MODEL = SON_OUTPUT / "models"

for p in [WORK_DATA, WORK_FIG, WORK_TABLE, WORK_PRED, WORK_MODEL]:
    p.mkdir(parents=True, exist_ok=True)

BEIJING_RAW = RAW_DIR / "beijing" / "PRSA_data_2010.1.1-2014.12.31.csv"

print("PROJECT_ROOT:", PROJECT_ROOT)
print("BEIJING_RAW:", BEIJING_RAW)
print("Working output:", SON_OUTPUT)

In [ ]:
df = pd.read_csv(BEIJING_RAW)
df['datetime'] = pd.to_datetime(df[['year','month','day','hour']])
df = df.sort_values('datetime').reset_index(drop=True)
df = df.drop(columns=['No'])
df['pm25_missing'] = df['pm2.5'].isna().astype('int8')

### Missing policy v1
Không interpolation hai phía cho feature forecasting. Ví dụ điểm thiếu ở t có thể bị nội suy bằng giá trị t+1, gây leakage. Giữ PM2.5 raw và để lag/rolling tự sinh NaN. Việc xử lý missing riêng cho health exposure sẽ được quy định khi aggregate daily.

In [ ]:
df['dayofweek'] = df['datetime'].dt.dayofweek
df['dayofyear'] = df['datetime'].dt.dayofyear
df['hour_sin'] = np.sin(2*np.pi*df['hour']/24)
df['hour_cos'] = np.cos(2*np.pi*df['hour']/24)
df['month_sin'] = np.sin(2*np.pi*df['month']/12)
df['month_cos'] = np.cos(2*np.pi*df['month']/12)
def season(m):
    if m in [3,4,5]: return 'spring'
    if m in [6,7,8]: return 'summer'
    if m in [9,10,11]: return 'autumn'
    return 'winter'
df['season'] = df['month'].map(season)
df['cbwd'] = df['cbwd'].astype('category')

In [ ]:
qc_cols=['pm2.5','DEWP','TEMP','PRES','Iws','Is','Ir']
qc=pd.DataFrame({'column':qc_cols,'min':[df[c].min() for c in qc_cols],'max':[df[c].max() for c in qc_cols],'missing':[df[c].isna().sum() for c in qc_cols]})
display(qc)
qc.to_csv(WORK_TABLE/'beijing_preprocessing_qc.csv', index=False)

In [ ]:
path=WORK_DATA/'beijing_hourly_clean_working.csv'
df.to_csv(path,index=False)
print('saved:',path)

## Promote approved result to MAIN

Chỉ chạy cell này sau khi Sơn/team đã kiểm tra kết quả.

```python
# Ví dụ: promote một file processed
source = WORK_DATA / "beijing_hourly_clean_working.csv"
target = PROCESSED_DIR / "beijing_hourly_clean.csv"
shutil.copy2(source, target)

# Ví dụ: promote figures/tables/models tương ứng
# shutil.copy2(WORK_FIG / "...png", MAIN_OUTPUT / "figures" / "...png")
# shutil.copy2(WORK_TABLE / "...csv", MAIN_OUTPUT / "tables" / "...csv")
# shutil.copy2(WORK_MODEL / "...pkl", MAIN_OUTPUT / "models" / "...pkl")
```

**Không có notebook nào tự động promote.**